# Data analysis

Data analysis on the train part of `split()` for each labelled dataset.

In [1]:
import random

import numpy as np
import torch
from sklearn.model_selection import train_test_split

SEED = 42
DATASETS = ("kryptonite-20", "kryptonite-30", "kryptonite-40", "kryptonite-40-adapt")


def load(name):
    """Load (X, y) arrays for a dataset."""
    return np.load(f"Datasets/{name}-X.npy"), np.load(f"Datasets/{name}-y.npy")


def set_seed(seed=42):
    """Set a seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def split(X, y):
    """60% train, 20% validation, 20% test."""
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=SEED)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=SEED)
    return X_train, y_train, X_val, y_val, X_test, y_test


set_seed(SEED)
train = {}
for name in DATASETS:
    X_train, y_train, *_ = split(*load(name))
    train[name] = (X_train.astype(np.float64), y_train.astype(np.float64))
    print(f"{name:<20} train X {X_train.shape}, positive rate {y_train.mean():.3f}")


kryptonite-20        train X (7200, 20), positive rate 0.495
kryptonite-30        train X (12000, 30), positive rate 0.505
kryptonite-40        train X (19200, 40), positive rate 0.497
kryptonite-40-adapt  train X (2400, 40), positive rate 0.516


In [2]:
print(f"{'dataset':<20} {'N':>6} {'mean range':>18} {'variance range':>16} "
      f"{'max change':>11} {'positive range':>16} {'negative range':>18} {'|x| range':>16}")
for name, (X, y) in train.items():
    N = len(X)
    mu, var = X.mean(0), X.var(0, ddof=1)
    change = np.abs((X - mu) / np.sqrt(var) - X).max()     # effect of standardising the train split
    pos, neg = X[X > 0], X[X < 0]                           # value ranges of positive / negative entries
    absX = np.abs(X)                                        # value range of absolute values
    print(f"{name:<20} {N:>6} [{mu.min():+.4f}, {mu.max():+.4f}] "
          f"[{var.min():.4f}, {var.max():.4f}] {change:>11.4f} "
          f"[{pos.min():.4f}, {pos.max():.4f}] [{neg.min():+.4f}, {neg.max():+.4f}] "
          f"[{absX.min():.4f}, {absX.max():.4f}]")

dataset                   N         mean range   variance range  max change   positive range     negative range        |x| range
kryptonite-20          7200 [-0.0245, +0.0142] [1.0060, 1.0143]      0.0328 [0.5910, 1.4326] [-1.4089, -0.5881] [0.5881, 1.4326]
kryptonite-30         12000 [-0.0128, +0.0168] [1.0064, 1.0158]      0.0222 [0.5001, 1.4720] [-1.4445, -0.5661] [0.5001, 1.4720]
kryptonite-40         19200 [-0.0131, +0.0153] [1.0065, 1.0134]      0.0235 [0.4933, 1.4369] [-1.4587, -0.5298] [0.4933, 1.4587]
kryptonite-40-adapt    2400 [-0.0381, +0.0543] [1.0009, 1.0181]      0.0590 [0.5872, 1.4729] [-1.3922, -0.5873] [0.5872, 1.4729]
